# ADTN Backtest Example

This notebook demonstrates running a backtest on ADTN using the simple backtester with candlestick pattern signals.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from data.yfinance_downloader import YFinanceDownloader
from backtest.simple_backtester import SimpleBacktester, BacktestConfig, generate_candlestick_signals

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

In [ ]:
# Download data
downloader = YFinanceDownloader(cache_dir="../data")
df = downloader.download_daily_full("ADTN")

# Use last 2 years for backtest
df = df[df["timestamp"] >= "2022-01-01"]
print(f"Data: {len(df)} rows from {df['timestamp'].min()} to {df['timestamp'].max()}")

In [ ]:
# Generate signals
signals = generate_candlestick_signals(df, lookback=5)

print(f"Long signals: {(signals == 1).sum()}")
print(f"Short signals: {(signals == -1).sum()}")
print(f"Flat: {(signals == 0).sum()}")

In [ ]:
# Run backtest
config = BacktestConfig(
    initial_capital=100000.0,
    position_size_pct=0.10,
    commission_per_share=0.005,
    slippage_pct=0.001,
)

backtester = SimpleBacktester(config)
metrics = backtester.run(df, signals)

print("=== BACKTEST METRICS ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Get trades
trades_df = backtester.get_trades_df()
print(f"Total trades: {len(trades_df)}")
if len(trades_df) > 0:
    print(trades_df.head(10))
    print(f"\nAverage PnL per trade: ${trades_df['pnl'].mean():.2f}")
    print(f"Best trade: ${trades_df['pnl'].max():.2f}")
    print(f"Worst trade: ${trades_df['pnl'].min():.2f}")

In [ ]:
# Plot equity curve
equity_df = backtester.get_equity_curve_df()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]}, sharex=True)

ax1.plot(equity_df["timestamp"], equity_df["equity"], label="Equity Curve", color="blue", linewidth=1.5)
ax1.set_ylabel("Equity ($)")
ax1.set_title("ADTN Backtest - Candlestick Pattern Strategy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df["timestamp"], df["close"], label="Close Price", color="black", linewidth=0.8, alpha=0.7)

if len(trades_df) > 0:
    longs = trades_df[trades_df["side"] == "long"]
    shorts = trades_df[trades_df["side"] == "short"]
    if not longs.empty:
        ax2.scatter(longs["entry_date"], longs["entry_price"], marker="^", color="green", s=60, label="Long Entry", zorder=5)
        ax2.scatter(longs["exit_date"], longs["exit_price"], marker="v", color="red", s=60, label="Long Exit", zorder=5)
    if not shorts.empty:
        ax2.scatter(shorts["entry_date"], shorts["entry_price"], marker="v", color="red", s=60, label="Short Entry", zorder=5)
        ax2.scatter(shorts["exit_date"], shorts["exit_price"], marker="^", color="green", s=60, label="Short Exit", zorder=5)

ax2.set_ylabel("Price ($)")
ax2.set_xlabel("Date")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly returns heatmap
equity_df["returns"] = equity_df["equity"].pct_change()
monthly = equity_df.set_index("timestamp") ["equity"].resample("M").last().pct_change().dropna()
monthly_pivot = monthly.groupby([monthly.index.year, monthly.index.month]).first().unstack()
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(12, 6))
sns.heatmap(monthly_pivot * 100, annot=True, fmt=".1f", cmap="RdYlGn", center=0, cbar_kws={"label": "Return (%)"})
plt.title("ADTN Strategy - Monthly Returns (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown chart
peak = equity_df["equity"].expanding().max()
drawdown = (equity_df["equity"] - peak) / peak * 100

plt.figure(figsize=(14, 4))
plt.fill_between(equity_df["timestamp"], drawdown, 0, color="red", alpha=0.3)
plt.plot(equity_df["timestamp"], drawdown, color="red", linewidth=0.5)
plt.ylabel("Drawdown (%)")
plt.title("ADTN Strategy - Drawdown")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max Drawdown: {drawdown.min():.2f}%")